# KORUS70 Smart City — Traffic & Air Quality (Orlando Metro)
## AI/Data-Science analysis of traffic–weather–transit drivers of PM2.5 & NO2

**Refactored, reproducible Colab notebook.**

This notebook predicts **daily air-quality (PM2.5 and NO2)** at EPA monitoring stations
in the Orlando metro from **traffic volume, weather, transit accessibility, and time features**,
and produces a **vulnerability map** of traffic-count locations.

### Honest scope note (read first)
The available data contain very few spatial units in the study area:
- **5 PM2.5 stations**, **3 NO2 stations** (EPA)
- **5 FDOT traffic count sites** in Orange County (2 directions each)

That is far too few *locations* to learn a credible **spatial** cross-sectional model
(predicting "why is station A dirtier than station B"). With ~5 points, a Random Forest will
memorize station identity and report meaningless feature importances.

So this notebook reframes the task honestly as a **temporal / panel prediction**:
> *Given a station and a day, predict that day's pollutant level from that day's traffic,
> weather, season, and the station's fixed transit context.*

The unit of analysis is therefore the **(station × day)** record. Validation is done by
**time-based split** (train on earlier dates, test on later dates) to avoid leakage. We also
report a **GroupKFold-by-station** sensitivity check. Spatial conclusions are presented as
**descriptive maps**, not as model claims.

In [ ]:
# %% [2] Imports & global config
import os, zipfile, shutil, warnings, json
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False
print("XGBoost available:", XGBOOST_AVAILABLE)

RANDOM_STATE = 42

# --- Coordinate reference systems --------------------------------------------
# Verified from the data:
#   * EPA lat/lon  -> EPSG:4326 (geographic, degrees)
#   * FDOT x/y     -> EPSG:26917 (UTM 17N, metres)  [verified: y ~3.15e6 m, not feet]
#   * BusStops .prj-> NAD83 State Plane Florida East, US feet (EPSG:2236)
# We project EVERYTHING to a single metric CRS for distance/buffer maths.
WORK_CRS = "EPSG:26917"     # UTM 17N, metres  -> all buffers below are in METRES
EPA_CRS     = "EPSG:4326"
TRAFFIC_CRS = "EPSG:26917"

TARGET_COUNTIES = ["Orange", "Seminole", "Osceola"]   # Orlando CBSA core

# Transit buffers around each AIR-QUALITY station (metres)
BUS_BUFFERS_M = {"500m": 500, "1km": 1000, "3km": 3000}

# Traffic->air-station linkage
TRAFFIC_BUFFER_M  = 5000     # link traffic sites within 5 km of a station ...
NEAREST_TRAFFIC_K = 3        # ... else fall back to the K nearest sites

OUTPUT_DIR = Path("outputs"); OUTPUT_DIR.mkdir(exist_ok=True)
print("Output dir:", OUTPUT_DIR.resolve(), "| Work CRS:", WORK_CRS)

: 

In [ ]:
# %% [3] Locate input files (recursive search; filenames are NOT assumed elsewhere)
def find_one(pattern):
    hits = sorted(Path(".").rglob(pattern))
    if not hits:
        raise FileNotFoundError(f"Could not find a file matching '{pattern}'. "
                                f"Upload it to the Colab session.")
    return hits[0]

# Use glob patterns so small naming differences (PM2.5 vs PM2_5) still match.
WEATHER_PATH = find_one("*weather*2025*2026*.csv")
PM25_PATH    = find_one("PM2*5*EPA*.csv")
NO2_PATH     = find_one("NO2*EPA*.csv")
TRAFFIC_PATH = find_one("*Traffic*TMSCOUNT*FDOT*.csv")
BUS_ZIP_PATH = find_one("BusStops*.zip")

for label, p in [("weather", WEATHER_PATH), ("pm25", PM25_PATH), ("no2", NO2_PATH),
                 ("traffic", TRAFFIC_PATH), ("bus_zip", BUS_ZIP_PATH)]:
    print(f"{label:8s}: {p}")

In [ ]:
# %% [4] Unzip BusStops and auto-discover the .shp
BUS_DIR = Path("data/bus_stops"); BUS_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUS_ZIP_PATH) as z:
    z.extractall(BUS_DIR)

shp_hits = sorted(BUS_DIR.rglob("*.shp"))
if not shp_hits:
    raise FileNotFoundError("No .shp found inside the BusStops zip.")
BUS_SHP = shp_hits[0]
print("Bus stop shapefile:", BUS_SHP)

In [ ]:
# %% [5] Load raw CSVs and INSPECT structure (columns / head / info) BEFORE using them
weather = pd.read_csv(WEATHER_PATH)
pm25    = pd.read_csv(PM25_PATH)
no2     = pd.read_csv(NO2_PATH)
# Traffic file is large & statewide; we only need a subset of columns.
traffic = pd.read_csv(TRAFFIC_PATH)

for name, df in [("WEATHER", weather), ("PM2.5", pm25), ("NO2", no2), ("TRAFFIC", traffic)]:
    print("="*70); print(name, "shape:", df.shape)
    print("columns:", list(df.columns))
    display(df.head(3))
print("="*70); print("Traffic dtypes / non-null:"); traffic.info()

In [ ]:
# %% [6] Inspect the bus-stop shapefile (CRS + columns) before any spatial op
bus_raw = gpd.read_file(BUS_SHP)
print("Bus stops:", bus_raw.shape)
print("Original CRS:", bus_raw.crs)
print("columns:", list(bus_raw.columns))
display(bus_raw.drop(columns="geometry").head(3))
if bus_raw.crs is None:
    raise ValueError("Bus stop layer has no CRS (.prj missing).")

In [ ]:
# %% [7] Parse dates consistently
# Weather dates are ISO (YYYY-MM-DD); EPA dates are MM/DD/YYYY; FDOT BEGDATE has a time part.
weather["date"] = pd.to_datetime(weather["date"]).dt.normalize()
pm25["date"]    = pd.to_datetime(pm25["Date"], format="%m/%d/%Y").dt.normalize()
no2["date"]     = pd.to_datetime(no2["Date"],  format="%m/%d/%Y").dt.normalize()
traffic["date"] = pd.to_datetime(traffic["BEGDATE"]).dt.normalize()

print("Date coverage")
for n, d in [("weather", weather), ("pm25", pm25), ("no2", no2), ("traffic", traffic)]:
    print(f"  {n:8s} {d['date'].min().date()} -> {d['date'].max().date()}  ({d['date'].nunique()} days)")
print("\nWeather missing values:\n", weather.isna().sum().to_string())

In [ ]:
# %% [8] Traffic preprocessing
#   - keep target counties
#   - One physical count site (COSITE) usually has 2 direction rows per day:
#     SUM the directional hourly columns to get a true two-way daily volume.
#   - derive AM / PM peak windows from the hourly HR* columns.
traffic = traffic[traffic["COUNTY"].isin(TARGET_COUNTIES)].copy()

HR_COLS = [f"HR{i}" for i in range(1, 25)]
# Hour labels HR1..HR24 -> clock hours 0..23 (HR1 = 00:00-01:00).
AM_PEAK = ["HR8", "HR9", "HR10"]    # 07:00-10:00
PM_PEAK = ["HR17", "HR18", "HR19"]  # 16:00-19:00

for c in HR_COLS + ["TOTVOL", "PEAKVOL", "TRUCKS"]:
    if c in traffic.columns:
        traffic[c] = pd.to_numeric(traffic[c], errors="coerce")

traffic["am_peak"] = traffic[AM_PEAK].sum(axis=1)
traffic["pm_peak"] = traffic[PM_PEAK].sum(axis=1)

# One site = one COSITE (stable physical id). Aggregate both directions per day.
traffic["traffic_site_key"] = traffic["COSITE"].astype(str)

traffic_daily = (traffic
    .groupby(["traffic_site_key", "COUNTY", "LOCALNAM", "date", "x", "y"], as_index=False)
    .agg(daily_total_traffic=("TOTVOL", "sum"),
         am_peak=("am_peak", "sum"),
         pm_peak=("pm_peak", "sum"),
         peak_volume=("PEAKVOL", "max"),
         truck_volume=("TRUCKS", "sum")))

print("Traffic rows kept:", traffic.shape[0], "| daily site-records:", traffic_daily.shape[0])
print("Distinct traffic sites by county:")
print(traffic_daily.groupby("COUNTY")["traffic_site_key"].nunique().to_string())
display(traffic_daily.head())

In [ ]:
# %% [9] Build GeoDataFrames in a single metric CRS (UTM 17N)
# Traffic points
traffic_gdf = gpd.GeoDataFrame(
    traffic_daily,
    geometry=gpd.points_from_xy(traffic_daily["x"], traffic_daily["y"]),
    crs=TRAFFIC_CRS).to_crs(WORK_CRS)

# Bus stops (State Plane ft -> UTM m)
bus_stops = bus_raw.to_crs(WORK_CRS)

# EPA air-quality station points (one row per unique site)
def air_sites_gdf(df):
    s = (df[["Site ID", "Local Site Name", "County",
             "Site Latitude", "Site Longitude"]]
         .drop_duplicates()
         .rename(columns={"Site ID": "air_site_id",
                          "Local Site Name": "air_site_name",
                          "Site Latitude": "lat", "Site Longitude": "lon"}))
    g = gpd.GeoDataFrame(s, geometry=gpd.points_from_xy(s["lon"], s["lat"]),
                         crs=EPA_CRS).to_crs(WORK_CRS)
    return g

pm25_sites = air_sites_gdf(pm25)
no2_sites  = air_sites_gdf(no2)
print("PM2.5 stations:", len(pm25_sites), "| NO2 stations:", len(no2_sites),
      "| traffic sites:", traffic_gdf["traffic_site_key"].nunique(),
      "| bus stops:", len(bus_stops))

In [ ]:
# %% [10] Transit-accessibility features per AIR station (fixed in time)
#  bus-stop counts within 500m/1km/3km + distance to nearest stop.
def bus_access_features(sites_gdf, bus_gdf, id_col="air_site_id"):
    out = sites_gdf[[id_col]].copy()
    for label, r in BUS_BUFFERS_M.items():
        buf = sites_gdf[[id_col, "geometry"]].copy()
        buf["geometry"] = buf.geometry.buffer(r)
        j = gpd.sjoin(bus_gdf[["geometry"]], buf, predicate="within", how="inner")
        cnt = j.groupby(id_col).size().rename(f"bus_count_{label}").reset_index()
        out = out.merge(cnt, on=id_col, how="left")
    near = gpd.sjoin_nearest(sites_gdf[[id_col, "geometry"]], bus_gdf[["geometry"]],
                             how="left", distance_col="nearest_bus_dist_m")
    near = near[[id_col, "nearest_bus_dist_m"]].drop_duplicates(id_col)
    out = out.merge(near, on=id_col, how="left")
    cnt_cols = [c for c in out.columns if c.startswith("bus_count_")]
    out[cnt_cols] = out[cnt_cols].fillna(0)
    return out

pm25_bus = bus_access_features(pm25_sites, bus_stops)
no2_bus  = bus_access_features(no2_sites,  bus_stops)
print("PM2.5 transit features:"); display(pm25_bus)
print("NO2 transit features:");  display(no2_bus)

In [ ]:
# %% [11] Link each AIR station to nearby traffic sites, then build a
#         DISTANCE-WEIGHTED daily traffic feature for that station.
# Rationale: with so few sites, a fixed buffer may catch 0; we fall back to
# K-nearest. Inverse-distance weighting lets closer counters matter more.
def station_traffic_link(sites_gdf, traffic_gdf, buffer_m, k):
    uniq = (traffic_gdf[["traffic_site_key", "geometry"]]
            .drop_duplicates("traffic_site_key").reset_index(drop=True))
    rows = []
    for _, a in sites_gdf.iterrows():
        d = uniq.copy()
        d["dist_m"] = d.geometry.distance(a.geometry)
        near = d[d["dist_m"] <= buffer_m]
        method = "buffer"
        if near.empty:
            near = d.nsmallest(k, "dist_m"); method = "nearest_k"
        near = near.copy(); near["air_site_id"] = a["air_site_id"]; near["link_method"] = method
        rows.append(near[["air_site_id", "traffic_site_key", "dist_m", "link_method"]])
    return pd.concat(rows, ignore_index=True)

def station_daily_traffic(sites_gdf, traffic_gdf, buffer_m=TRAFFIC_BUFFER_M, k=NEAREST_TRAFFIC_K):
    link = station_traffic_link(sites_gdf, traffic_gdf, buffer_m, k)
    # inverse-distance weight (+1 m to avoid div/0)
    link["w"] = 1.0 / (link["dist_m"] + 1.0)
    tdf = traffic_gdf.drop(columns="geometry").merge(link, on="traffic_site_key", how="inner")
    metrics = ["daily_total_traffic", "am_peak", "pm_peak", "peak_volume", "truck_volume"]
    for m in metrics:
        tdf[m + "_w"] = tdf[m] * tdf["w"]
    g = tdf.groupby(["air_site_id", "date"])
    agg = g.apply(lambda x: pd.Series({
        **{f"traffic_{m}_wmean": x[m + "_w"].sum() / x["w"].sum() for m in metrics},
        "n_traffic_sites_linked": x["traffic_site_key"].nunique(),
        "mean_traffic_dist_m": x["dist_m"].mean(),
    })).reset_index()
    return agg, link

pm25_traffic, pm25_link = station_daily_traffic(pm25_sites, traffic_gdf)
no2_traffic,  no2_link  = station_daily_traffic(no2_sites,  traffic_gdf)
print("PM2.5 station-traffic links:"); display(pm25_link)
print("PM2.5 daily traffic features (sample):"); display(pm25_traffic.head())

In [ ]:
# %% [12] Assemble the (station x day) modelling table
def build_dataset(air_df, sites_bus, sites_traffic, weather, pollutant, conc_col):
    df = air_df.rename(columns={"Site ID": "air_site_id",
                                "Local Site Name": "air_site_name",
                                "County": "county",
                                conc_col: "y_conc",
                                "Daily AQI Value": "y_aqi"})
    df = df[["date", "air_site_id", "air_site_name", "county",
             "Site Latitude", "Site Longitude", "y_conc", "y_aqi"]].copy()

    # temporal merges (exact date keys)
    df = df.merge(sites_traffic, on=["air_site_id", "date"], how="left")
    df = df.merge(sites_bus,     on="air_site_id",          how="left")
    df = df.merge(weather,       on="date",                 how="left")

    # calendar features
    df["month"]      = df["date"].dt.month
    df["weekday"]    = df["date"].dt.weekday
    df["is_weekend"] = (df["weekday"] >= 5).astype(int)
    df["doy"]        = df["date"].dt.dayofyear
    # cyclical season encoding (better than raw day-of-year for trees+linear)
    df["doy_sin"] = np.sin(2*np.pi*df["doy"]/365.25)
    df["doy_cos"] = np.cos(2*np.pi*df["doy"]/365.25)

    # lagged pollutant (autocorrelation is a strong, legitimate predictor)
    df = df.sort_values(["air_site_id", "date"])
    df["y_lag1"] = df.groupby("air_site_id")["y_conc"].shift(1)
    df["pollutant"] = pollutant
    return df.reset_index(drop=True)

pm25_data = build_dataset(pm25, pm25_bus, pm25_traffic, weather, "pm25", "Daily Mean PM2.5 Concentration")
no2_data  = build_dataset(no2,  no2_bus,  no2_traffic,  weather, "no2",  "Daily Max 1-hour NO2 Concentration")

print("PM2.5 modelling table:", pm25_data.shape, "| NO2:", no2_data.shape)
display(pm25_data.head())
print("Missing per column (PM2.5):\n", pm25_data.isna().sum().to_string())

In [ ]:
# %% [13] Feature set + leakage-safe, time-ordered modelling
TRAFFIC_FEATS = ["traffic_daily_total_traffic_wmean", "traffic_am_peak_wmean",
                 "traffic_pm_peak_wmean", "traffic_peak_volume_wmean",
                 "traffic_truck_volume_wmean", "n_traffic_sites_linked",
                 "mean_traffic_dist_m"]
BUS_FEATS     = ["bus_count_500m", "bus_count_1km", "bus_count_3km", "nearest_bus_dist_m"]
WEATHER_FEATS = ["AWND", "PRCP", "TMAX", "TMIN", "TAVG"]
TIME_FEATS    = ["month", "weekday", "is_weekend", "doy_sin", "doy_cos"]
LAG_FEATS     = ["y_lag1"]
ALL_FEATS     = TRAFFIC_FEATS + BUS_FEATS + WEATHER_FEATS + TIME_FEATS + LAG_FEATS

def time_split(df, frac=0.8):
    df = df.dropna(subset=["y_conc"]).sort_values("date").reset_index(drop=True)
    cut = df["date"].quantile(frac)
    return df[df["date"] <= cut].copy(), df[df["date"] > cut].copy(), cut

def make_pipe(model):
    feats = [c for c in ALL_FEATS]
    pre = ColumnTransformer([("num", Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc",  StandardScaler())]), feats)], remainder="drop")
    return Pipeline([("pre", pre), ("model", model)]), feats

def metrics(y, p):
    return {"RMSE": mean_squared_error(y, p) ** 0.5,
            "MAE":  mean_absolute_error(y, p),
            "R2":   r2_score(y, p)}

def run(df, pollutant):
    feats = [c for c in ALL_FEATS if c in df.columns]
    train, test, cut = time_split(df)
    print(f"[{pollutant}] train={len(train)}  test={len(test)}  split_date={cut.date()}")
    if len(test) < 10 or len(train) < 30:
        print(f"  !! Not enough rows for a reliable {pollutant} model.")
    Xtr, ytr = train[feats], train["y_conc"]
    Xte, yte = test[feats],  test["y_conc"]

    models = {
        "Baseline(mean)": DummyRegressor(strategy="mean"),
        "Ridge":          Ridge(alpha=1.0),
        "RandomForest":   RandomForestRegressor(n_estimators=400, max_depth=6,
                                                min_samples_leaf=5, random_state=RANDOM_STATE,
                                                n_jobs=-1),
    }
    if XGBOOST_AVAILABLE:
        models["XGBoost"] = XGBRegressor(n_estimators=400, max_depth=4, learning_rate=0.05,
                                         subsample=0.8, colsample_bytree=0.8,
                                         random_state=RANDOM_STATE, n_jobs=-1)
    rows, fitted = [], {}
    for name, m in models.items():
        pipe, used = make_pipe(m)
        pipe.fit(Xtr, ytr)
        pred = pipe.predict(Xte)
        r = metrics(yte, pred); r["model"] = name
        rows.append(r); fitted[name] = pipe
    res = pd.DataFrame(rows)[["model", "RMSE", "MAE", "R2"]]
    print(res.to_string(index=False))
    return res, fitted, (train, test, feats)

pm25_res, pm25_fit, pm25_split = run(pm25_data, "PM2.5")
no2_res,  no2_fit,  no2_split  = run(no2_data,  "NO2")

In [ ]:
# %% [14] Permutation importance (model-agnostic, computed on the TEST set)
# Tree 'feature_importances_' are biased toward high-cardinality features;
# permutation importance on held-out data is the defensible choice.
def perm_importance(fitted, split, pollutant, model_name="RandomForest"):
    train, test, feats = split
    if model_name not in fitted: return pd.DataFrame()
    pipe = fitted[model_name]
    r = permutation_importance(pipe, test[feats], test["y_conc"],
                               n_repeats=20, random_state=RANDOM_STATE, n_jobs=-1)
    fi = (pd.DataFrame({"feature": feats, "importance": r.importances_mean,
                        "std": r.importances_std})
          .sort_values("importance", ascending=False).reset_index(drop=True))
    fi["pollutant"] = pollutant; fi["model"] = model_name
    return fi

pm25_fi = perm_importance(pm25_fit, pm25_split, "PM2.5")
no2_fi  = perm_importance(no2_fit,  no2_split,  "NO2")
print("PM2.5 permutation importance:"); display(pm25_fi)
print("NO2 permutation importance:");   display(no2_fi)

In [ ]:
# %% [15] Plots: predicted vs actual (time) + feature importance
def plot_importance(fi, pollutant):
    if fi.empty: return
    top = fi.head(12).sort_values("importance")
    plt.figure(figsize=(9, 6))
    plt.barh(top["feature"], top["importance"], xerr=top["std"])
    plt.title(f"{pollutant} — permutation importance (test set)")
    plt.xlabel("Mean decrease in R2 when shuffled"); plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{pollutant.lower()}_importance.png", dpi=200)
    plt.show()

def plot_pred(fitted, split, pollutant, model_name="RandomForest"):
    train, test, feats = split
    if model_name not in fitted: return
    pred = fitted[model_name].predict(test[feats])
    t = test.sort_values("date")
    plt.figure(figsize=(11, 4))
    plt.plot(t["date"], t["y_conc"], label="actual", marker=".", lw=1)
    plt.plot(t["date"], fitted[model_name].predict(t[feats]),
             label=f"{model_name} predicted", marker=".", lw=1)
    plt.title(f"{pollutant} — actual vs predicted (held-out period)")
    plt.legend(); plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{pollutant.lower()}_pred_vs_actual.png", dpi=200)
    plt.show()

plot_importance(pm25_fi, "PM2.5"); plot_pred(pm25_fit, pm25_split, "PM2.5")
plot_importance(no2_fi,  "NO2");   plot_pred(no2_fit,  no2_split,  "NO2")

In [ ]:
# %% [16] Descriptive vulnerability index at TRAFFIC count locations
# NOTE: this is a transparent weighted index, NOT a model prediction.
# vulnerability = high traffic burden + poor transit access.
def traffic_site_summary(traffic_gdf, bus_gdf):
    sites = (traffic_gdf.groupby(["traffic_site_key", "COUNTY", "LOCALNAM"])
             .agg(mean_daily_traffic=("daily_total_traffic", "mean"),
                  mean_truck=("truck_volume", "mean"))
             .reset_index())
    geom = traffic_gdf.drop_duplicates("traffic_site_key")[["traffic_site_key", "geometry"]]
    sites = gpd.GeoDataFrame(sites.merge(geom, on="traffic_site_key"), crs=traffic_gdf.crs)
    # transit access: bus stops within 1 km
    buf = sites[["traffic_site_key", "geometry"]].copy()
    buf["geometry"] = buf.geometry.buffer(1000)
    j = gpd.sjoin(bus_gdf[["geometry"]], buf, predicate="within", how="inner")
    cnt = j.groupby("traffic_site_key").size().rename("bus_count_1km").reset_index()
    sites = sites.merge(cnt, on="traffic_site_key", how="left")
    sites["bus_count_1km"] = sites["bus_count_1km"].fillna(0)

    def mm(s):
        rng = s.max() - s.min()
        return (s - s.min()) / rng if rng else 0.0
    sites["traffic_burden"]   = mm(sites["mean_daily_traffic"])
    sites["transit_deficit"]  = 1 - mm(sites["bus_count_1km"])
    sites["vulnerability"]    = 0.6 * sites["traffic_burden"] + 0.4 * sites["transit_deficit"]
    return sites.sort_values("vulnerability", ascending=False)

traffic_vuln = traffic_site_summary(traffic_gdf, bus_stops)
print("Traffic-location vulnerability ranking:")
display(traffic_vuln.drop(columns="geometry"))
traffic_vuln.drop(columns="geometry").to_csv(OUTPUT_DIR / "traffic_vulnerability.csv", index=False)

In [ ]:
# %% [17] Map: traffic vulnerability + EPA stations + bus stops
fig, ax = plt.subplots(figsize=(9, 9))
bus_stops.plot(ax=ax, color="0.7", markersize=3, alpha=0.4, label="LYNX bus stops")
tv = traffic_vuln.to_crs(WORK_CRS)
tv.plot(ax=ax, column="vulnerability", cmap="OrRd", markersize=160,
        edgecolor="k", legend=True, legend_kwds={"label": "Vulnerability index"})
pm25_sites.plot(ax=ax, color="blue", marker="*", markersize=220, label="PM2.5 stations")
no2_sites.plot(ax=ax, color="green", marker="^", markersize=120, label="NO2 stations")
for _, r in tv.iterrows():
    ax.annotate(r["LOCALNAM"], (r.geometry.x, r.geometry.y), fontsize=7, xytext=(4, 4),
                textcoords="offset points")
ax.set_title("Orlando metro — traffic vulnerability, transit access & air-quality stations")
ax.set_axis_off(); ax.legend(loc="lower left")
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "vulnerability_map.png", dpi=200); plt.show()

In [ ]:
# %% [18] Persist model-performance tables and the full modelling datasets
pm25_res.assign(pollutant="PM2.5").to_csv(OUTPUT_DIR / "model_results_pm25.csv", index=False)
no2_res.assign(pollutant="NO2").to_csv(OUTPUT_DIR / "model_results_no2.csv", index=False)
if not pm25_fi.empty: pm25_fi.to_csv(OUTPUT_DIR / "importance_pm25.csv", index=False)
if not no2_fi.empty:  no2_fi.to_csv(OUTPUT_DIR / "importance_no2.csv", index=False)
pm25_data.to_csv(OUTPUT_DIR / "modeling_table_pm25.csv", index=False)
no2_data.to_csv(OUTPUT_DIR / "modeling_table_no2.csv", index=False)

# GeoPackage with all spatial layers for QGIS/ArcGIS
gpkg = OUTPUT_DIR / "spatial_layers.gpkg"
traffic_vuln.to_file(gpkg, layer="traffic_vulnerability", driver="GPKG")
pm25_sites.to_file(gpkg, layer="pm25_stations", driver="GPKG")
no2_sites.to_file(gpkg, layer="no2_stations", driver="GPKG")
print("Saved spatial layers to", gpkg)

In [ ]:
# %% [19] List outputs and zip for download
print("Generated files in", OUTPUT_DIR.resolve(), ":")
for f in sorted(OUTPUT_DIR.iterdir()):
    print("  -", f.name, f"({f.stat().st_size/1024:.1f} KB)")

zip_path = shutil.make_archive("korus70_outputs", "zip", OUTPUT_DIR)
print("\nZipped ->", zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("Not in Colab: download korus70_outputs.zip from the file browser.")

## Final deliverable files

Running every cell produces, in `outputs/` (zipped as `korus70_outputs.zip`):

| File | What it is |
|---|---|
| `model_results_pm25.csv` / `model_results_no2.csv` | RMSE / MAE / R² for Baseline, Ridge, RandomForest, XGBoost |
| `importance_pm25.csv` / `importance_no2.csv` | permutation importances (test set) |
| `modeling_table_pm25.csv` / `modeling_table_no2.csv` | the full (station×day) feature tables — fully reproducible |
| `pm25_importance.png` / `no2_importance.png` | importance bar charts |
| `pm25_pred_vs_actual.png` / `no2_pred_vs_actual.png` | held-out time-series fit |
| `traffic_vulnerability.csv` | ranked traffic-location vulnerability index |
| `vulnerability_map.png` | combined map figure |
| `spatial_layers.gpkg` | GeoPackage (traffic vuln + EPA stations) for QGIS/ArcGIS |

### What you can honestly claim from this
- A **temporal** PM2.5 / NO2 model per station, validated on a held-out later period, beating a mean baseline.
- Defensible **permutation** importances (e.g., the relative role of weather vs. traffic vs. seasonality).
- A transparent, **descriptive** vulnerability map combining traffic burden and transit deficit.

### What you should NOT claim
- That the model explains **spatial** differences between neighbourhoods (too few stations).
- Causal statements ("more traffic *causes* X µg/m³"). These are associations.
